# GestuRhythm v2 - Training
Train Gesture Emotion Encoder + Conditioned Decoder
Input: gesture sequences (N, 30, 225) + emotion labels (N, 2)

In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Ho tro nhieu duong dan dataset
import os, glob
POSSIBLE_PATHS = [
    '/kaggle/input/datasets/quangleuq/v3-data/',
    '/kaggle/input/gesturhythm-v2/',
    '/kaggle/input/v3-data/',
]
DATA_DIR = None
for p in POSSIBLE_PATHS:
    if os.path.exists(os.path.join(p, 'sequences.npy')):
        DATA_DIR = p; break
assert DATA_DIR, f'Khong tim thay dataset! Da thu: {POSSIBLE_PATHS}'
print(f'Dataset: {DATA_DIR}')

X = np.load(DATA_DIR + 'sequences.npy')   # (N, 30, 225)
y = np.load(DATA_DIR + 'emotions.npy')    # (N, 2) valence, arousal
m = np.load(DATA_DIR + 'mode_ids.npy')    # (N,)
print(f'X: {X.shape} | y: {y.shape}')
print(f'Phan bo mode: {dict(zip(*np.unique(m, return_counts=True)))}')


In [ ]:
# === Gesture Emotion Encoder ===
class GestureEmotionEncoder(nn.Module):
    def __init__(self, input_dim=225, d_model=128, nhead=4, num_layers=3):
        super().__init__()
        self.embed   = nn.Linear(input_dim, d_model)
        self.pos_enc = nn.Embedding(512, d_model)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Linear(64, 2), nn.Tanh()
        )
    def forward(self, x):
        B, T, _ = x.shape
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        x    = self.embed(x) + self.pos_enc(pos)
        x    = self.transformer(x, mask=mask)
        return self.head(x[:, -1, :])

model  = GestureEmotionEncoder().to(device)
total  = sum(p.numel() for p in model.parameters())
print(f'Params: {total:,}')

In [ ]:
class GestureDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X, self.y, self.aug = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), augment
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i] + torch.randn_like(self.X[i]) * 0.01 if self.aug else self.X[i]
        return x, self.y[i]

ds      = GestureDataset(X, y, augment=True)
n_train = int(0.8 * len(ds))
train_ds, val_ds = random_split(ds, [n_train, len(ds)-n_train],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
print(f'Train: {n_train} | Val: {len(ds)-n_train}')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)
criterion = nn.MSELoss()

EPOCHS = 80
train_losses, val_losses = [], []
val_mae_v_list, val_mae_a_list = [], []

for epoch in range(EPOCHS):
    model.train()
    tl = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()
    model.eval()
    vl, all_pred, all_true = 0, [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred   = model(xb)
            vl    += criterion(pred, yb).item()
            all_pred.append(pred.cpu().numpy())
            all_true.append(yb.cpu().numpy())
    import numpy as np_
    preds_np = np_.vstack(all_pred); trues_np = np_.vstack(all_true)
    mae_v = np_.abs(preds_np[:,0] - trues_np[:,0]).mean()
    mae_a = np_.abs(preds_np[:,1] - trues_np[:,1]).mean()
    scheduler.step()
    train_losses.append(tl / len(train_loader))
    val_losses.append(vl / len(val_loader))
    val_mae_v_list.append(float(mae_v))
    val_mae_a_list.append(float(mae_a))
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f} | '
              f'MAE_val={mae_v:.4f} | MAE_aro={mae_a:.4f}')
print('Done!')


In [ ]:
# Bieu do
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('GestuRhythm v2 - Emotion Encoder Training', fontweight='bold')

# Loss curve
axes[0].plot(train_losses, label='Train', color='steelblue')
axes[0].plot(val_losses,   label='Val',   color='coral')
axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Pred vs True valence
model.eval()
preds, trues = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds.append(model(xb.to(device)).cpu().numpy())
        trues.append(yb.numpy())
preds = np.vstack(preds); trues = np.vstack(trues)

axes[1].scatter(trues[:,0], preds[:,0], alpha=0.4, s=15, color='steelblue')
axes[1].plot([-1,1],[-1,1],'r--'); axes[1].set_title('Valence: Pred vs True')
axes[1].set_xlabel('True'); axes[1].set_ylabel('Predicted'); axes[1].grid(True, alpha=0.3)

axes[2].scatter(trues[:,1], preds[:,1], alpha=0.4, s=15, color='coral')
axes[2].plot([-1,1],[-1,1],'r--'); axes[2].set_title('Arousal: Pred vs True')
axes[2].set_xlabel('True'); axes[2].set_ylabel('Predicted'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()

mae_v = np.abs(preds[:,0] - trues[:,0]).mean()
mae_a = np.abs(preds[:,1] - trues[:,1]).mean()
print(f'MAE Valence: {mae_v:.4f} | MAE Arousal: {mae_a:.4f}')

In [ ]:
# Emotion space visualization
fig, ax = plt.subplots(figsize=(7, 6))
mood_names = {0:'HAPPY_HIGH', 1:'HAPPY_LOW', 2:'SAD_HIGH', 3:'SAD_LOW', 4:'NEUTRAL'}
colors     = ['#2ecc71','#3498db','#e74c3c','#9b59b6','#f39c12']

# Ve vung cam xuc
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.text( 0.6,  0.8, 'HAPPY_HIGH', fontsize=9, alpha=0.5)
ax.text( 0.6, -0.8, 'HAPPY_LOW',  fontsize=9, alpha=0.5)
ax.text(-0.9,  0.8, 'SAD_HIGH',   fontsize=9, alpha=0.5)
ax.text(-0.9, -0.8, 'SAD_LOW',    fontsize=9, alpha=0.5)

mode_ids = np.load('/kaggle/input/gesturhythm-v2/mode_ids.npy') if True else None
for i in range(min(len(preds), 200)):
    ax.scatter(preds[i,0], preds[i,1], alpha=0.6, s=20, color='steelblue')

ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
ax.set_xlabel('Valence (buon <-> vui)'); ax.set_ylabel('Arousal (thu gian <-> nang luong)')
ax.set_title('Predicted Emotion Space')
ax.grid(True, alpha=0.3)
plt.savefig('emotion_space.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
torch.save({
    'model_state': model.state_dict(),
    'config': {'input_dim':225,'d_model':128,'nhead':4,'num_layers':3},
    'val_loss': val_losses[-1],
    'mae_valence': float(mae_v),
    'mae_arousal': float(mae_a),
}, 'gesture_emotion_encoder.pt')
print('Saved: gesture_emotion_encoder.pt')
print(f'Final val loss: {val_losses[-1]:.4f}')